In [ ]:
# TODO
# Make notebook runable
# Update paths
# Update input data

In [1]:
import pandas as pd
import pickle
%matplotlib inline

#Import style guide to make plots
import sys
sys.path.append('../../../../tjn_tools/')

from style_guide import *
#Import some data_processing tools to convert names to ISO3 
from data_processing import *

import seaborn as sns
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'

import pylab as plt
import matplotlib as mpl
%matplotlib inline


#Avoid overlapping text|
# from adjustText import adjust_text

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

Note: Originally this was named 98.combine_parts.ipynb, but I renamed it to 98.combine_parts.ipynb to make it run last.

In [2]:
path_files = "../data/raw/"
path_files_final = "../data/final/"
path_files_temp = "../data/processing/"
path_figures = "../results/"

year_cbcr = 2017

In [3]:
#Files from the data portal
country_level = "../../../../Final data/20210810_country-level-data.csv"
country_year_level = "../../../../Final data/20210810_country-year-level-data.csv"
bilateral_level = "../../../../Final data/20210810_bilateral-year-level-data.csv"

#File ufrom the IFF analysis (202004-Risk-based%20IFF/Scripts/Analysis_2021_04.ipynb)
iff_file_path = f"{path_files_final}iff_sotj_table.csv"

#Salary nurses OECD
nurses_file = f"{path_files}oecd_nurses_clean.xlsx"

#Not sure why relative paths don't work for this file
tax_evasion_file_output = "~/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2021 Report/Offshore wealth/Price of offshore full results.xlsx"

In [4]:
info_expenditures_output = f"{path_files_final}{year_cbcr}_info_expenditures.csv"
cols_other_info = ['who_gvt_health_expenditure','Govt_exp_educ_gdp_wb',
        'total_taxes_revenue', 'cit_revenue', 'iso3', 'GDP_int', 'POP_int',
        'region_tjn',"UKt","OECD","OECD_OCT","G20","EU28","month_wage","FSI2020_Rank","FSI2020_Share","FSI2020_Score","CTHI21_Rank","CTHI21_Share","CTHI21_Score"]


In [5]:
etr_output = f"{path_files_final}{year_cbcr}_cbcr_etr_rates.xlsx"
df_etrs = pd.read_excel(etr_output,index_col=0)
iso3_to_etr = df_etrs["ETR_total"].to_dict()

In [6]:
# Input
file_output = f"{year_cbcr}_tax_avoidance_sotj_table.xlsx"
sotj_table_output = f"{path_files_final}{file_output}"

In [7]:
iso3_to_cit_output = f"{path_files_final}{year_cbcr}_iso3_to_cit.dump"
iso3_to_cit = pickle.load(open(iso3_to_cit_output,"rb+"))


In [8]:
# Output
file_output = f"{year_cbcr}combined_output.xlsx"
workstream_path = f"../../../../../Workstreams/Scale of Tax Injustice/State of Tax Justice report/2021 Report/Combined_offshore_corporate/"
final_table_output = f"{path_files_final}{file_output}"
final_table_output_workstream = f"{workstream_path}{file_output}"

# 1. Read data

In [9]:
#Some variables were not added before: "EU28 OECT", "EU27", "EU27 OCT", "GBR OCT", th_eu_blacklist_201006, th_eu_greylist_201006, th_unctad2015
countryYearBase = pd.read_stata('../../../../../Workstreams/Financial Secrecy/CTHI/CTHI-2021/GSW/210211 countryYearBase for CTHI2021.dta')
countryYearBase = countryYearBase.loc[:,["country","EU27","EU27_OCT","EU28_OCT","GBR_OCT","th_eu_blacklist_201006", "th_eu_greylist_201006", "th_unctad2015"]]
countryYearBase["iso3"] = countryYearBase["country"].apply(get_iso3, print_failure = False)
countryYearBase = countryYearBase.loc[countryYearBase["country"] != "West Bank and Gaza"]
countryYearBase = countryYearBase.drop(columns=["country"])
countryYearBase = countryYearBase.drop_duplicates()
countryYearBase["iso3"].value_counts().head(2)

YUG    1
NZL    1
Name: iso3, dtype: int64

In [10]:
#Class and other info file
other_info = pd.read_csv(info_expenditures_output, sep="\t", usecols=cols_other_info)
other_info = pd.merge(other_info,countryYearBase,how="left")
other_info.head()

,iso3,Govt_exp_educ_gdp_wb,total_taxes_revenue,cit_revenue,who_gvt_health_expenditure,month_wage,GDP_int,POP_int,FSI2020_Rank,FSI2020_Share,...,G20,UKt,OECD_OCT,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015
0,ABW,1.807188e+08,6.412902e+08,NaN,NaN,2188.935759,2.966542e+09,1.047770e+05,112.0,0.002251,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,AFG,6.913388e+08,1.535079e+09,2.363759e+08,9.528032e+07,89.595632,1.947716e+10,3.522628e+07,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AGO,NaN,2.902758e+10,NaN,1.491610e+09,411.463465,1.114437e+11,2.890116e+07,35.0,0.010145,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AIA,NaN,5.842218e+07,0.000000e+00,NaN,1730.294149,2.997023e+08,1.500911e+04,62.0,0.005668,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
4,ALA,NaN,NaN,NaN,NaN,3661.086201,1.560000e+09,2.948900e+04,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
#Other estimates of tax avoidance
cols = {"iso3":"iso3","ps_sr_cobham2018":"TA: C&J short-run 2018 (USD million)","ps_lr_cobham2018":"TA: C&J long-run 2018 (USD million)","ps_torslov2020_2016": "TA: TWZ 2016 (USD million)","ps_torslov2020_2017": "TA: TWZ 2017 (USD million)","ps_jansky2019": "TA: JP 2019 (USD million)"}
ext_estimates = pd.read_csv(country_level,skiprows=1,sep="\t",usecols=(list(cols.keys())))
# cols = [_ for _ in ohter_m.columns if ("ps_" in _) and not ("_so") in _ and (_ not in ("ps_cobham2018","ps_torslov2018") )]
ext_estimates = ext_estimates[cols].dropna(thresh=2).drop_duplicates(subset=["iso3"])
ext_estimates = ext_estimates.rename(columns=cols)
ext_estimates[list(ext_estimates.columns )[1:]] /= 1E6
ext_estimates.loc[ext_estimates["iso3"]=="IND"]

,iso3,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million)
46,IND,20431.862,137229.74,9588.894258,11342.488388,71598.17


In [12]:
other_info = pd.merge(other_info,ext_estimates,how="left",validate="1:1")
other_info.head()

,iso3,Govt_exp_educ_gdp_wb,total_taxes_revenue,cit_revenue,who_gvt_health_expenditure,month_wage,GDP_int,POP_int,FSI2020_Rank,FSI2020_Share,...,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million)
0,ABW,1.807188e+08,6.412902e+08,NaN,NaN,2188.935759,2.966542e+09,1.047770e+05,112.0,0.002251,...,1.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN
1,AFG,6.913388e+08,1.535079e+09,2.363759e+08,9.528032e+07,89.595632,1.947716e+10,3.522628e+07,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,AGO,NaN,2.902758e+10,NaN,1.491610e+09,411.463465,1.114437e+11,2.890116e+07,35.0,0.010145,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,AIA,NaN,5.842218e+07,0.000000e+00,NaN,1730.294149,2.997023e+08,1.500911e+04,62.0,0.005668,...,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
4,ALA,NaN,NaN,NaN,NaN,3661.086201,1.560000e+09,2.948900e+04,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
#Income class
income_class = pd.read_csv(country_year_level,skiprows=1,sep="\t",usecols=["iso3","year","IncomeClass","GDP_int","POP_int"])#
income_class["GDPpc"] = income_class["GDP_int"]/income_class["POP_int"]

income_class["IncomeClassInt"] = pd.cut(income_class["GDPpc"],[0,1046,4096,12696,np.inf],labels=["L","LM","UM","H"])

missing_income = set(income_class["iso3"]) - set(income_class.dropna(subset=["IncomeClass"])["iso3"])
print(missing_income)

info_on_income_class = income_class.loc[~income_class["iso3"].isin(missing_income)].dropna(subset=["IncomeClass"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = income_class.loc[income_class["iso3"].isin(missing_income)].dropna(subset=["IncomeClassInt"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = no_info_on_income_class.rename(columns = {"IncomeClass": "IncomeClass_na","IncomeClassInt": "IncomeClass"})

income_class = pd.concat([info_on_income_class, no_info_on_income_class])
income_class["IncomeClass"] = income_class["IncomeClass"].map({'H':"High income", 'L':"Low income", 'LM':"Lower-middle income", 'UM':"Upper-middle income"})

income_class = income_class.loc[:,["iso3","IncomeClass"]]

other_info = pd.merge(other_info,income_class,how="left",validate="1:1")
other_info.head()

{nan, 'GBA', 'JEY', 'ATA', 'NFK', 'SPM', 'COK', 'GLP', 'AIA', 'GUF', 'MTQ', 'PCN', 'MSR', 'IRS_Other_Europe', 'IOT', 'GGY', 'WLD', 'SJM', 'BLM', 'ATF', 'CCK', 'IRS_Other_Asia_Oceania', 'BES', 'TKL', 'Foreign_controlled_US_comps', 'SCG', 'WSH', 'FLK', 'WBG', 'IRS_Other_Africa', 'EAZ', 'BVT', 'Stateless', 'VAT', 'KSV', 'REU', 'PUS', 'SHN', 'ALA', 'NIU', 'ESH', 'CXR', 'IRS_Other_America', 'WLF', 'SGS', 'HMD'}


,iso3,Govt_exp_educ_gdp_wb,total_taxes_revenue,cit_revenue,who_gvt_health_expenditure,month_wage,GDP_int,POP_int,FSI2020_Rank,FSI2020_Share,...,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million),IncomeClass
0,ABW,1.807188e+08,6.412902e+08,NaN,NaN,2188.935759,2.966542e+09,1.047770e+05,112.0,0.002251,...,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,High income
1,AFG,6.913388e+08,1.535079e+09,2.363759e+08,9.528032e+07,89.595632,1.947716e+10,3.522628e+07,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Low income
2,AGO,NaN,2.902758e+10,NaN,1.491610e+09,411.463465,1.114437e+11,2.890116e+07,35.0,0.010145,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Lower-middle income
3,AIA,NaN,5.842218e+07,0.000000e+00,NaN,1730.294149,2.997023e+08,1.500911e+04,62.0,0.005668,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,High income
4,ALA,NaN,NaN,NaN,NaN,3661.086201,1.560000e+09,2.948900e+04,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,High income


In [14]:
#Data on nurses salaries
oecd_nurses = pd.read_excel(nurses_file,
              skiprows=4,na_values=[".."],
              usecols=["Year","2010","2011","2012","2013","2014","2015","2016","2017","2018","2019"])

oecd_nurses = pd.melt(oecd_nurses,id_vars="Year",value_vars=["2010","2011","2012","2013","2014","2015","2016","2017","2018","2019"],var_name="date",value_name="month_wage_nurse")
oecd_nurses = oecd_nurses.dropna()
oecd_nurses = oecd_nurses.sort_values(by=["Year","date"],ascending=False).drop_duplicates(subset=["Year"])
oecd_nurses["month_wage_nurse"] /= 12 #Make it monthy

#Change columns
oecd_nurses["iso3"] = oecd_nurses["Year"].apply(get_iso3)
oecd_nurses = oecd_nurses.dropna().drop(columns=["Year","date"])
other_info = pd.merge(other_info,oecd_nurses,how="left")

#Replace with OECD data when available
other_info.loc[~np.isnan(other_info["month_wage_nurse"]),"month_wage"] = other_info.loc[~np.isnan(other_info["month_wage_nurse"]),"month_wage_nurse"]
other_info.head()

,iso3,Govt_exp_educ_gdp_wb,total_taxes_revenue,cit_revenue,who_gvt_health_expenditure,month_wage,GDP_int,POP_int,FSI2020_Rank,FSI2020_Share,...,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million),IncomeClass,month_wage_nurse
0,ABW,1.807188e+08,6.412902e+08,NaN,NaN,2188.935759,2.966542e+09,1.047770e+05,112.0,0.002251,...,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,High income,NaN
1,AFG,6.913388e+08,1.535079e+09,2.363759e+08,9.528032e+07,89.595632,1.947716e+10,3.522628e+07,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Low income,NaN
2,AGO,NaN,2.902758e+10,NaN,1.491610e+09,411.463465,1.114437e+11,2.890116e+07,35.0,0.010145,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Lower-middle income,NaN
3,AIA,NaN,5.842218e+07,0.000000e+00,NaN,1730.294149,2.997023e+08,1.500911e+04,62.0,0.005668,...,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,High income,NaN
4,ALA,NaN,NaN,NaN,NaN,3661.086201,1.560000e+09,2.948900e+04,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,High income,NaN


In [15]:
#IFF file
iff_file = pd.read_csv(iff_file_path,sep=",")#,header=None)
iff_file["ISO-3 code of country"] = iff_file["Name"].apply(get_iso3, print_failure=False)
iff_file = iff_file.dropna(subset=["ISO-3 code of country"])
del iff_file["Name"]
iff_file.head()

,Outward Banking Positions,Inward Banking Positions,Outward FDI,Inward FDI,Outward Portfolio Inv.,Inward Portfolio Inv.,Outward Trade (Exports),Inward Trade (Imports),Top Flow,Top Vulnerability,Vulnerability Region,Top1,Top2,Top3,ISO-3 code of country
1,NaN,NaN,NaN,47.0,NaN,NaN,55.0,58.0,Outward Trade (Electronics),63.0,61.0,Egypt (26.2%),Qatar (18.2%),Tunisia (11.7%),DZA
2,NaN,NaN,NaN,NaN,NaN,86.0,61.0,57.0,Inward Portfolio Inv.,86.0,60.0,United States (31.1%),Luxembourg (26.0%),Netherlands (8.0%),AGO
3,NaN,NaN,NaN,NaN,NaN,NaN,68.0,60.0,Outward Trade (Textiles),69.0,59.0,Bangladesh (41.4%),Vietnam (17.5%),Malaysia (12.7%),BEN
4,NaN,NaN,58.0,57.0,NaN,NaN,57.0,57.0,Outward Trade (Other),66.0,60.0,South Africa (47.9%),United States (8.1%),United Kingdom (7.8%),BWA
5,NaN,NaN,NaN,NaN,NaN,NaN,66.0,NaN,Outward Trade (Exports),66.0,57.0,United States (45.9%),Singapore (35.2%),Ghana (9.0%),IOT


In [16]:

# x = tax_avoidance_file.dropna(subset=["MNCs"])
# x = x.loc[x["Revenue loss using CIT (M)"]>0]
# x = x.groupby("ISO-3 code of country").sum()["Revenue loss using CIT (M)"]
# y = df_merged.loc[df_merged["TA: Tax revenue loss using CIT (USD million)"]>0].groupby("ISO-3 code of country").sum()["TA: Tax revenue loss using CIT (USD million)"]

# z = pd.concat([x,y],axis=1)
# z.loc[z["Revenue loss using CIT (M)"].round(0) != z["TA: Tax revenue loss using CIT (USD million)"].round(0)].sort_values(by="Revenue loss using CIT (M)")

# tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD","CIT"]

In [17]:
#Tax avoidance
tax_avoidance_file = pd.read_excel(sotj_table_output).drop_duplicates()
tax_avoidance_file["CIT"] /= 100
tax_avoidance_file["ETR"] /= 100
tax_avoidance_file["ISO-3 code of country"] = tax_avoidance_file["Name"].map(get_iso3).replace("SCG","SRB") #wrong in the first file

tax_avoidance_file.head()

 Africa not matched to any file
 Asia not matched to any file
 Caribbean/American isl. not matched to any file
 Europe not matched to any file
 Latin America not matched to any file
 Northern America not matched to any file
 Oceania not matched to any file
nan not matched to any file
nan not matched to any file


,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),...,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
0,Africa,NaN,0.2732,0.2896,34334.0,21771.0,47482.0,11301.4,8115.1,14730.6,...,61.0,683.0,6.984814e+12,3.398176e+09,0.5,47.8,10.0,0.6,10.0,NaN
1,Algeria,Domestic,0.2500,0.1491,808.0,163.0,1467.0,202.1,40.9,366.7,...,0.0,0.0,1.800524e+11,4.058320e+07,0.4,10.1,NaN,3.2,19.0,DZA
2,Algeria,Foreign (in),0.2500,0.1491,-3806.0,-4207.0,-3399.0,-951.6,-1051.8,-849.7,...,1.0,11.0,1.800524e+11,4.058320e+07,-2.1,-47.6,NaN,-14.8,-93.0,DZA
3,Algeria,Foreign (out),0.2500,0.1491,847.0,755.0,935.0,211.7,188.7,233.7,...,1.0,11.0,1.800524e+11,4.058320e+07,0.5,10.6,NaN,3.3,20.0,DZA
4,Angola,Domestic,0.3000,0.5432,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.114437e+11,2.890116e+07,0.0,0.0,NaN,0.0,0.0,AGO


In [18]:
#Tax evasion
tax_evasion_file = pd.read_excel(tax_evasion_file_output)
tax_evasion_file["ISO-3 code of country"] = tax_evasion_file["Country"].map(get_iso3)
tax_evasion_file = tax_evasion_file.drop(columns=["Country"])
tax_evasion_file.head()


Alderney not matched to any file


,ISO-3 code of country,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries
0,USA,0.198868,1977.196777,0.092249,36578.140625,0.116565,19900.470703
1,GBR,0.113913,1132.548950,0.400079,25482.351562,0.176684,30164.238281
2,IRL,0.057535,572.025024,1.435121,13728.600586,0.055368,9452.647461
3,LUX,0.045225,449.638031,6.323585,10292.214844,0.089662,15307.429688
4,CHN,0.044760,445.017761,0.031164,10012.899414,0.000000,0.000000


In [19]:
#other_info.loc[other_info["iso3"].isin([get_iso3(_) for _ in ["Malaysia","Vietnam","Thailand","Cambodia","Indonesia","Myanmar","Philippines"]]),["iso3","month_wage"]]

In [20]:
1977.196777*1000*0.05

98859.83885

In [21]:
ta_dom = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Domestic",["ISO-3 code of country","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"N_reporters"]]   
ta_dom.columns = list(ta_dom.columns[:1]) + [_ + " - dom" for _ in ta_dom.columns[1:]]
ta_fo = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (out)",["ISO-3 code of country","ETR","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust","N_reporters"]]
ta_fo.columns = list(ta_fo.columns[:2]) + [_ + " - for lose" for _ in ta_fo.columns[2:]]
ta_ga = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (in)",["ISO-3 code of country","CIT","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust"]]
ta_ga.columns = list(ta_ga.columns[:2]) + [_ + " - for gain" for _ in ta_ga.columns[2:]]
       
ta = pd.concat([ta_dom.set_index("ISO-3 code of country"),ta_fo.set_index("ISO-3 code of country"),ta_ga.set_index("ISO-3 code of country")],axis=1,sort=False)
ta = ta.reset_index().rename(columns={"index":"ISO-3 code of country"})

for tax in ["CIT","ETR"]:
    for var in ["Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)']:
        for end in ["- dom","- for lose","- for gain"]:
            ta[f"Revenue loss using {tax} (M) {end}"] = ta[f"{var} {end}"]*ta[tax]

ta = ta.rename(columns={"N_reporters - dom":"Reporting country"})
ta.head()

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,...,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom
0,DZA,808.0,163.0,1467.0,0.0,0.1491,847.0,755.0,935.0,233.75,...,11.0,0.250,-3806.0,-4207.0,-3399.0,-849.75,-506.7909,2.0,366.75,218.7297
1,AGO,0.0,0.0,0.0,0.0,0.5432,601.0,520.0,692.0,207.60,...,9.0,0.300,0.0,0.0,0.0,0.00,0.0000,1.0,0.00,0.0000
2,BEN,0.0,0.0,0.0,0.0,0.1745,211.0,169.0,265.0,51.41,...,4.0,0.194,0.0,0.0,0.0,0.00,0.0000,0.0,0.00,0.0000
3,BWA,0.0,0.0,0.0,0.0,0.1462,23.0,20.0,27.0,5.94,...,7.0,0.220,-209.0,-243.0,-183.0,-40.26,-26.7546,2.0,0.00,0.0000
4,BFA,0.0,0.0,0.0,0.0,0.0392,0.0,0.0,3.0,0.48,...,5.0,0.160,-320.0,-431.0,-245.0,-39.20,-9.6040,2.0,0.00,0.0000


In [22]:
def return_gain(common_var = "Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1):
    vals = ((ta["{}{}".format(common_var,vars_[0])]+sign*ta["{}{}".format(common_var,vars_[0])].abs())/2).fillna(0)
    print(vals)
    for v in vars_[1:]:
        vals += ((ta["{}{}".format(common_var,v)]+sign*ta["{}{}".format(common_var,v)].abs())/2).fillna(0)
    print(vals)
    return vals

for v in ["dom","for lose","for gain"]:
    for st in ["","Min. ","Max. "]:
        ta[f"{st}Profit loss (M) - {v}"] = ta[f"{st}Profit loss (M) - {v}"].fillna(0)
        
ta["ETR"] = ta["ISO-3 code of country"].map(iso3_to_etr)
ta["CIT"] = ta["ISO-3 code of country"].map(iso3_to_cit)
for st in ["","Min. ","Max. "]:
    ta[f"{st}Profit gain (M)"] = -return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1)
    ta[f"{st}Profit loss (M)"] = return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=1)

for st in ["","Min. ","Max. "]:
    ta[f"{st}Revenue gain using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue gain using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue loss using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit loss (M)"]
    ta[f"{st}Revenue loss using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit loss (M)"]

ta["Robust"] = ta["Reporting country"]+((ta["N_reporters - for lose"])>3).astype(int)

ta.sort_values(by="Profit gain (M)").tail(20)

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
209    0.0
210    0.0
211    0.0
212    0.0
213    0.0
Name: Profit loss (M) - dom, Length: 214, dtype: float64
0     -3806.0
1         0.0
2         0.0
3      -209.0
4      -320.0
        ...  
209   -1406.0
210       0.0
211      -9.0
212       0.0
213     -21.0
Name: Profit loss (M) - dom, Length: 214, dtype: float64
0      808.0
1        0.0
2        0.0
3        0.0
4        0.0
       ...  
209     75.0
210      0.0
211      0.0
212      0.0
213      0.0
Name: Profit loss (M) - dom, Length: 214, dtype: float64
0      1655.0
1       601.0
2       211.0
3        23.0
4         0.0
        ...  
209     213.0
210     249.0
211       1.0
212       5.0
213      18.0
Name: Profit loss (M) - dom, Length: 214, dtype: float64
0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
       ...  
209   -138.0
210      0.0
211      0.0
212      0.0
213      0.0
Name: Min. Profit loss (M) - dom, Length: 214, dtype: flo

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,...,Revenue loss using ETR (M),Min. Revenue gain using CIT (M),Min. Revenue gain using ETR (M),Min. Revenue loss using CIT (M),Min. Revenue loss using ETR (M),Max. Revenue gain using CIT (M),Max. Revenue gain using ETR (M),Max. Revenue loss using CIT (M),Max. Revenue loss using ETR (M),Robust
148,IMN,119.0,118.0,120.0,1.0,0.005949,1330.0,1105.0,1631.0,0.0000,...,8.620771,0.000000,96.303944,0.000000,7.276192,0.000000,64.962178,0.000000,10.417508,2.0
150,JEY,0.0,0.0,0.0,0.0,0.008953,511.0,446.0,608.0,121.6000,...,4.574747,3911.000059,175.066866,89.200001,3.992832,2886.400043,129.203017,121.600002,5.443143,1.0
92,TWN,-16998.0,-30856.0,-4825.0,0.0,0.131396,865.0,831.0,907.0,154.1900,...,113.657818,5571.410066,4306.251643,141.270002,109.190343,1100.240013,850.396992,154.190002,119.176464,1.0
77,MYS,-19379.0,-19490.0,-19255.0,1.0,0.187334,4369.0,4161.0,4618.0,1108.3200,...,818.461777,4677.599903,3651.137568,998.639979,779.496327,4621.199904,3607.114103,1108.319977,865.107916,2.0
162,NOR,-19823.0,-19923.0,-19690.0,1.0,0.265060,3172.0,2982.0,3383.0,811.9200,...,840.770689,4781.519900,5280.792695,715.679985,790.409267,4725.599902,5219.033688,811.919983,896.698373,2.0
171,SWE,-9809.0,-9983.0,-9583.0,1.0,0.109020,4728.0,4490.0,5010.0,1102.2000,...,515.445719,4913.479978,2434.848706,987.799996,489.499001,4609.439979,2284.183312,1102.199995,546.189308,2.0
89,KOR,-23509.0,-24099.0,-22770.0,1.0,0.229926,276.0,264.0,291.0,64.0200,...,63.459589,5301.779976,5540.987775,58.080000,60.700476,5009.399977,5235.416060,64.020000,66.908479,2.0
177,BRA,-26671.0,-27016.0,-26291.0,1.0,0.226875,23138.0,22560.0,23845.0,8107.3000,...,5249.429853,9185.440108,6129.250450,7670.400090,5118.296201,8938.940105,5964.766197,8107.300095,5409.830359,2.0
118,PRI,0.0,0.0,0.0,0.0,0.016005,1399.0,1285.0,1557.0,607.2300,...,22.391334,15035.669460,617.050095,501.149982,20.566736,12485.849552,512.407822,607.229978,24.920162,1.0
147,IRL,-4896.0,-4939.0,-4840.0,1.0,0.105862,647.0,620.0,678.0,84.7500,...,68.492648,4693.750000,3975.114292,77.500000,65.634377,4356.125000,3689.181304,84.750000,71.774367,2.0


In [23]:
ta.loc[ta["ISO-3 code of country"]=="USA",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
196,9943.83,0.0,70620.39,76910.85


In [24]:
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
9,NaN,NaN,NaN,1947.049967


In [25]:
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Profit loss" in _) and ("." not in _) ]]

,Profit loss (M) - dom,Profit loss (M) - for lose,Profit loss (M) - for gain,Profit loss (M)
9,0.0,5563.0,0.0,5563.0


In [26]:
## MERGE FILES
df_merged = pd.merge(tax_evasion_file,ta,how="outer").dropna(subset=["ISO-3 code of country"])
df_merged = pd.merge(df_merged,iff_file,how="outer").dropna(subset=["ISO-3 code of country"])
# df_merged = pd.merge(df_merged,childrens_file,how="left").dropna(subset=["ISO-3 code of country"])
df_merged = pd.merge(df_merged,other_info,left_on="ISO-3 code of country",right_on="iso3",how="left")

df_merged["Offshore wealth owned by citizens of country (USD billion)"] *= 1000*0.05 #Convert to million and multiply by the rate of return
df_merged["FSI2020_Share"] *= 100
df_merged["CTHI21_Share"] *= 100

df_merged["CIT"] = df_merged["ISO-3 code of country"].map(iso3_to_cit)
df_merged.head()


,ISO-3 code of country,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,...,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million),IncomeClass,month_wage_nurse
0,USA,0.198868,98859.838867,0.092249,36578.140625,0.116565,19900.470703,249043.0,239890.0,261557.0,...,0.0,0.0,0.0,98168.14000,482521.58000,152059.622039,161894.205354,80188.80,High income,6472.500000
1,GBR,0.113913,56627.447510,0.400079,25482.351562,0.176684,30164.238281,42794.0,41616.0,44401.0,...,0.0,0.0,0.0,935.74630,4599.43270,80524.551152,95967.955094,13601.63,High income,3925.994167
2,IRL,0.057535,28601.251221,1.435121,13728.600586,0.055368,9452.647461,-4896.0,-4939.0,-4840.0,...,0.0,0.0,0.0,NaN,NaN,-117108.667912,-126229.304737,NaN,High income,4783.339167
3,LUX,0.045225,22481.901550,6.323585,10292.214844,0.089662,15307.429688,8937.0,8886.0,9016.0,...,0.0,0.0,0.0,156.78277,770.62746,-50117.882747,-66046.640961,NaN,High income,9065.615000
4,CHN,0.044760,22250.888062,0.031164,10012.899414,0.000000,0.000000,-51230.0,-55861.0,-45984.0,...,0.0,0.0,0.0,39787.03000,267227.90000,50425.959162,51398.595430,NaN,Upper-middle income,NaN


In [27]:
tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD"]

,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),...,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
27,Chad,Domestic,0.35,0.375,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.143972e+10,14578733.0,0.0,0.0,0.0,0.0,0.0,TCD
28,Chad,Foreign (out),0.35,0.375,5563.0,5530.0,5608.0,1947.1,1935.4,1962.8,...,0.0,3.0,1.143972e+10,14578733.0,48.6,5234.1,1942.2,304.6,381.0,TCD


In [28]:
rename_cols = {}
for st in ["","Min. ","Max. "]:
    rename_cols[f"{st}Profit gain (M)"] = f"TA: {st}Tax base gain (USD million)"
    rename_cols[f"{st}Profit loss (M)"] = f"TA: {st}Tax base loss (USD million)"
    rename_cols[f"{st}Revenue gain using CIT (M)"] = f"TA: {st}Tax revenue gain using CIT (USD million)"
    rename_cols[f"{st}Revenue gain using ETR (M)"] = f"TA: {st}Tax revenue gain using ETR (USD million)"
    rename_cols[f"{st}Revenue loss using CIT (M)"] = f"TA: {st}Tax revenue loss using CIT (USD million)"
    rename_cols[f"{st}Revenue loss using ETR (M)"] = f"TA: {st}Tax revenue loss using ETR (USD million)"
    

rename_cols.update({"Offshore wealth owned by citizens of country (USD billion)": 'OW: Tax base loss (USD million)',
 "Tax revenue loss: Offshore wealth (USD million)": 'OW: Tax revenue loss (USD million)',
 'Share of global tax loss inflicted by country': 'Harm OW: Total (% total)',
 'Tax loss inflicted on other countries': 'Harm OW: Total (USD million)',
 'Reporting country': 'TA: Reporting country',
 'Robust': 'TA: Robust',
 'N_reporters - for lose': 'TA: Number countries reporting',
 'Outward Banking Positions': 'IFF: Outward Banking Positions',
 'Inward Banking Positions': 'IFF: Inward Banking Positions',
 'Outward FDI': 'IFF: Outward FDI',
 'Inward FDI': 'IFF: Inward FDI',
 'Outward Portfolio Inv.': 'IFF: Outward Portfolio Inv.',
 'Inward Portfolio Inv.': 'IFF: Inward Portfolio Inv.',
 'Outward Trade (Exports)': 'IFF: Outward Trade (Exports)',
 'Inward Trade (Imports)': 'IFF: Inward Trade (Imports)',
 'Top Flow': 'IFF: Top Flow',
 'Top Vulnerability': 'IFF: Top Vulnerability',
 'Vulnerability Region': 'IFF: Vulnerability Region',
 'Top1': 'IFF: Top1',
 'Top2': 'IFF: Top2',
 'Top3': 'IFF: Top3',
 'FSI2020_Rank': 'FSI_Rank',
 'FSI2020_Share': 'FSI_Share',
 'FSI2020_Score': 'FSI_Score',
 'CTHI21_Rank': 'CTHI_Rank',
 'CTHI21_Share': 'CTHI_Share',
 'CTHI21_Score': 'CTHI_Score',
 #'Children lives lost due to tax revenue loss (total)': 'Children lives lost due to tax revenue loss (total)',
 'Govt_exp_educ_gdp_wb': 'WBD: Government education expenditure',
 'who_gvt_health_expenditure': 'WHO: Government health expenditure',
 'total_taxes_revenue': 'GRD: Total tax revenue',
 'cit_revenue': 'GRD: Total corporate income revenue',
 'GDP_int': 'GDP',
 'POP_int': 'POP',
 'region_tjn': 'Region',
 'IncomeClass': 'Income Class',
 'UKt': 'UK territory',
 'month_wage': 'Average wage'})

In [29]:
df_merged = df_merged.rename(columns = rename_cols)
df_merged = df_merged.dropna(subset=["OW: Tax base loss (USD million)","Harm OW: Total (USD million)","TA: Tax base loss (USD million)","TA: Tax base gain (USD million)"],how="all")
df_merged["Country"] = df_merged["ISO-3 code of country"].map(iso3_to_name)
df_merged.loc[df_merged["ISO-3 code of country"]=="PUS","Region"] = 'Caribean/American isl.'
df_merged["IncomeClass2"] = df_merged["Income Class"].str.contains("Low").replace({False: "Higher", True: "Lower"})
df_merged.head()

,ISO-3 code of country,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,...,th_unctad2015,TA: C&J short-run 2018 (USD million),TA: C&J long-run 2018 (USD million),TA: TWZ 2016 (USD million),TA: TWZ 2017 (USD million),TA: JP 2019 (USD million),Income Class,month_wage_nurse,Country,IncomeClass2
0,USA,0.198868,98859.838867,0.092249,36578.140625,0.116565,19900.470703,249043.0,239890.0,261557.0,...,0.0,98168.14000,482521.58000,152059.622039,161894.205354,80188.80,High income,6472.500000,United States,Higher
1,GBR,0.113913,56627.447510,0.400079,25482.351562,0.176684,30164.238281,42794.0,41616.0,44401.0,...,0.0,935.74630,4599.43270,80524.551152,95967.955094,13601.63,High income,3925.994167,United Kingdom,Higher
2,IRL,0.057535,28601.251221,1.435121,13728.600586,0.055368,9452.647461,-4896.0,-4939.0,-4840.0,...,0.0,NaN,NaN,-117108.667912,-126229.304737,NaN,High income,4783.339167,Ireland,Higher
3,LUX,0.045225,22481.901550,6.323585,10292.214844,0.089662,15307.429688,8937.0,8886.0,9016.0,...,0.0,156.78277,770.62746,-50117.882747,-66046.640961,NaN,High income,9065.615000,Luxembourg,Higher
4,CHN,0.044760,22250.888062,0.031164,10012.899414,0.000000,0.000000,-51230.0,-55861.0,-45984.0,...,0.0,39787.03000,267227.90000,50425.959162,51398.595430,NaN,Upper-middle income,NaN,China,Higher


In [30]:
#Impute missing values
df_merged["health_gdp"] = df_merged["WHO: Government health expenditure"]/df_merged["GDP"]
df_merged["educ_gdp"] = df_merged["WBD: Government education expenditure"]/df_merged["GDP"]
df_merged["tax_rev_gdp"] = df_merged["GRD: Total tax revenue"]/df_merged["GDP"]
df_merged["c_tax_rev_gdp"] = df_merged["GRD: Total corporate income revenue"]/df_merged["GDP"]

reg_av = df_merged.groupby("Income Class").sum()
reg_av["health_gdp"] = reg_av["WHO: Government health expenditure"]/reg_av["GDP"]
reg_av["educ_gdp"] = reg_av["WBD: Government education expenditure"]/reg_av["GDP"]
reg_av["tax_rev_gdp"] = reg_av["GRD: Total tax revenue"]/reg_av["GDP"]
reg_av["c_tax_rev_gdp"] = reg_av["GRD: Total corporate income revenue"]/reg_av["GDP"]

reg_av = reg_av.to_dict()

for v in ["health_gdp","educ_gdp","tax_rev_gdp","c_tax_rev_gdp"]:
    df_merged.loc[np.isnan(df_merged[v]),v] =  df_merged.loc[np.isnan(df_merged[v]),"Income Class"].map(reg_av[v])

df_merged["WHO: Government health expenditure (imp)"] = df_merged["health_gdp"]*df_merged["GDP"]
df_merged["WBD: Government education expenditure (imp)"] = df_merged["educ_gdp"]*df_merged["GDP"]
df_merged["GRD: Total tax revenue (imp)"] = df_merged["tax_rev_gdp"]*df_merged["GDP"]
df_merged["GRD: Total corporate income revenue (imp)"] = df_merged["c_tax_rev_gdp"]*df_merged["GDP"]

In [31]:
#Tax losses
df_merged["Loss OW (USD million)"] = df_merged['OW: Tax revenue loss (USD million)'].fillna(0)
df_merged.loc[df_merged["Loss OW (USD million)"]<0,"Loss OW (USD million)"] = 0
for st in ["","Min. ","Max. "]:
    df_merged[f"{st}Loss TA using CIT (USD million)"] = df_merged['TA: Tax revenue loss using CIT (USD million)'].fillna(0)
    df_merged[f"{st}Loss TA using ETR (USD million)"] = df_merged['TA: Tax revenue loss using ETR (USD million)'].fillna(0)


df_merged["GDP (only lossers)"] = df_merged["GDP"].copy()
df_merged.loc[df_merged["GDP (only lossers)"]<0,"GDP (only lossers)"] = 0

for tax in ["CIT","ETR"]:
    df_merged[f"Loss Total using {tax} (USD million)"] = df_merged[f"Loss TA using {tax} (USD million)"].fillna(0) + df_merged["Loss OW (USD million)"].fillna(0)
    df_merged[f"Loss Total using {tax} (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GDP"]
    df_merged[f"Loss Total using {tax} (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} Global (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GDP"].sum()
    df_merged[f"Loss Total using {tax} Regional (% GDP)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GDP"].transform(sum)
    df_merged[f"Loss Total using {tax} Global (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GRD: Total tax revenue (imp)"].sum()
    df_merged[f"Loss Total using {tax} Regional (% gvt tax revenue)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GRD: Total tax revenue (imp)"].transform(sum)
    df_merged[f"Loss Total using {tax} (per capita)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["POP"]
    df_merged[f"Loss Total using {tax} (% Education)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged['WBD: Government education expenditure']
    df_merged[f"Loss Total using {tax} (% Health)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["WHO: Government health expenditure"]
    df_merged[f"Loss Total using {tax} (%  gvt corporate revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total corporate income revenue"]
    df_merged[f"Loss Total using {tax} (%  gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} (# Nurses)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["Average wage"]/12


In [40]:
df_merged.loc[df_merged["ISO-3 code of country"]=="USA",[f"Loss Total using CIT (% gvt tax revenue)","Loss Total using CIT (USD million)","Loss TA using CIT (USD million)","Loss OW (USD million)"]]

,Loss Total using CIT (% gvt tax revenue),Loss Total using CIT (USD million),Loss TA using CIT (USD million),Loss OW (USD million)
0,3.086932,113488.990625,76910.85,36578.140625


In [33]:
df_merged["TA: Tax revenue loss using ETR (USD million)"].sum()
print("TRL {0:2,.0f}B (95% CI {1:2,.0f}-{2:2,.0f}B)".format(*1e-3*df_merged[["TA: Tax revenue loss using CIT (USD million)","TA: Min. Tax revenue loss using CIT (USD million)","TA: Max. Tax revenue loss using CIT (USD million)"]].sum()))

TRL 312B (95% CI 294-338B)


In [34]:
#Harm to others
#OW: Already in the file
df_merged["Harm OW: Total (% total)"] *= 100

for st in ["","Min. ","Max. "]:
    #Total tax lost
    #OW already in the file
    df_merged[f"{st}Base gain TA (USD million)"] = df_merged[f'TA: {st}Tax base gain (USD million)'].fillna(0)
    df_merged[f"{st}Harm TA: Total (% total)"] = 100*df_merged[f"{st}Base gain TA (USD million)"]/df_merged[f"{st}Base gain TA (USD million)"].sum()
    
    total_loss_ta_cit = df_merged[f"{st}Loss TA using CIT (USD million)"].sum()
    total_loss_ta_etr = df_merged[f"{st}Loss TA using ETR (USD million)"].sum()
    df_merged[f"{st}Harm TA: Total using CIT (USD million)"] = total_loss_ta_cit*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100
    df_merged[f"{st}Harm TA: Total using ETR (USD million)"] = total_loss_ta_etr*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100

for tax in ["CIT","ETR"]:
    total_loss = df_merged[f"Loss Total using {tax} (USD million)"].sum()
    df_merged[f"Harm: Total using {tax} (USD million)"] = df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]
    df_merged[f"Harm: Total using {tax} (% total)"] = 100*df_merged[f"Harm: Total using {tax} (USD million)"]/(df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]).sum()

    total_nurses = df_merged[f'Loss Total using {tax} (# Nurses)'].sum()
    df_merged[f"Harm: Total using {tax} (# Nurses)"] = total_nurses*df_merged[f"Harm: Total using {tax} (% total)"]/100

df_merged["Year"] = 2021

#Add missing values
df_merged.loc[np.isnan(df_merged["OW: Tax base loss (USD million)"]),
              ['OW: Tax base loss (USD million)',
 'OW: Tax revenue loss (USD million)',
 'Harm OW: Total (% total)',
 'Harm OW: Total (USD million)','Loss OW (USD million)']] = np.NaN

cond = np.isnan(df_merged["TA: Tax base loss (USD million)"]) & np.isnan(df_merged["OW: Tax base loss (USD million)"])
df_merged.loc[cond,['Loss Total (USD million)',
 'Loss Total (% GDP)',
 'Loss Total (% gvt tax revenue)',
 'Loss Total Global (% GDP)',
 'Loss Total Regional (% GDP)',
 'Loss Total Global (% gvt tax revenue)',
 'Loss Total Regional (% gvt tax revenue)',
 'Loss Total (per capita)',
 'Loss Total (% Education)',
 'Loss Total (% Health)',
 'Loss Total (%  gvt corporate revenue)',
 'Loss Total (%  gvt tax revenue)',
 'Loss Total (# Nurses)']] = np.nan

In [35]:

df_merged = pd.merge(df_merged,other_info,how="outer")
df_merged.loc[df_merged["Country"]=="India"]

,ISO-3 code of country,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,...,POP_int,FSI2020_Rank,FSI2020_Share,FSI2020_Score,CTHI21_Rank,CTHI21_Share,CTHI21_Score,region_tjn,UKt,IncomeClass
45,IND,0.001236,614.466858,0.004281,220.470703,0.0,0.0,25646.0,24805.0,26605.0,...,1.323848e+09,47.0,0.00701,47.8375,NaN,NaN,NaN,Asia,0.0,Lower-middle income


In [36]:
def robust(s):
    return a

a = []
for i,row in df_merged.iterrows():
    if row["TA: Robust"]==2:
        a.append("background-color: #44b0c6")
    elif row["TA: Robust"]==1:
        a.append("background-color: #94c9d4")
    else:
        a.append("background-color: white")
        
    
# df_merged.style.apply(robust)

In [37]:



df_merged.style.apply(robust).to_excel("~/Downloads/combined_output.xlsx",index=None)
df_merged.style.apply(robust).to_excel(final_table_output,index=None)
df_merged.style.apply(robust).to_excel(final_table_output_workstream,index=None)



In [38]:
df_merged.loc[df_merged["Income Class"].fillna("X")=="X"]

,ISO-3 code of country,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,...,POP_int,FSI2020_Rank,FSI2020_Share,FSI2020_Score,CTHI21_Rank,CTHI21_Share,CTHI21_Score,region_tjn,UKt,IncomeClass
154,BES,2.435582e-05,1.210757e+01,NaN,2.542590e+00,0.0,0.0,NaN,NaN,NaN,...,2.423704e+04,NaN,NaN,NaN,NaN,NaN,NaN,Caribbean/American isl.,0.0,NaN
215,ATF,1.600331e-09,7.955436e-04,NaN,1.670642e-04,0.0,0.0,NaN,NaN,NaN,...,2.500000e+02,NaN,NaN,NaN,NaN,NaN,NaN,Africa,0.0,NaN
216,BVT,5.063657e-10,2.517205e-04,NaN,5.286130e-05,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Caribbean/American isl.,0.0,NaN
218,CCK,6.833145e-13,3.396839e-07,NaN,7.133361e-08,0.0,0.0,NaN,NaN,NaN,...,5.960000e+02,NaN,NaN,NaN,NaN,NaN,NaN,Oceania,0.0,NaN
221,IOT,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.0,0.0,NaN,NaN,NaN,...,2.500000e+03,NaN,NaN,NaN,NaN,NaN,NaN,Africa,1.0,NaN
235,SJM,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.926000e+03,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,NaN
236,NFK,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.748000e+03,NaN,NaN,NaN,NaN,NaN,NaN,Oceania,0.0,NaN
237,EAZ,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
240,FCZ,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241,CXR,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.205000e+03,NaN,NaN,NaN,NaN,NaN,NaN,Oceania,0.0,NaN


In [34]:
## Work for ATP (danish people)
#Main results
final_data_output = f"{path_files_temp}{year_cbcr}_replicates.csv"
atp_prbook = pd.read_csv(final_data_output,sep="\t").reset_index(drop=True)
atp_prbook = atp_prbook.dropna()
atp_prbook = atp_prbook.loc[atp_prbook["profits"]>0]
atp_prbook = atp_prbook.groupby(["iso3_d","n_rep"]).sum()[["profits"]].groupby("iso3_d").median().reset_index()
atp_prbook.head()



,iso3_d,profits
0,ABW,1.032280e+08
1,AFG,5.516194e+07
2,AGO,3.816572e+09
3,ALB,7.490495e+07
4,ARE,2.772194e+10


In [36]:
atp_other = pd.read_excel(final_table_output_workstream)
atp_other["Gain - Loss"] = atp_other["TA: Tax base gain (USD million)"] -  atp_other["TA: Tax base loss (USD million)"]
atp_other = atp_other[['ISO-3 code of country','ETR', "Gain - Loss"]]
atp_other.columns = ["iso3_d","ETR","Gain - Loss"]
atp_other = atp_other.loc[atp_other["Gain - Loss"]>0]
atp_other = pd.merge(atp_other,atp_prbook,how="left")
atp_other["profits"] /= 1E6

In [43]:
atp_other["var"] = 100*atp_other["Gain - Loss"]/atp_other["profits"] * atp_other["Gain - Loss"]/atp_other["Gain - Loss"].sum()
atp_other=  atp_other.sort_values(by="var",ascending=False)
atp_other.to_excel("C:/Users/javga/Downloads/temp.xlsx")

In [44]:
atp_other.loc[atp_other["profits"]<atp_other["Gain - Loss"]]

,iso3_d,ETR,Gain - Loss,profits,var
